# Complex CNN — đánh giá model đã train

Notebook này **không train lại model**. Nó tải code, dataset và checkpoint epoch 187 từ GitHub, sau đó đánh giá toàn bộ 2.325 ảnh test.

Trên Colab, chọn **Runtime → Change runtime type → T4 GPU** rồi chạy lần lượt các cell từ trên xuống.

In [ ]:
# Tải code, dataset và checkpoint từ GitHub.
%cd /content
!git clone --depth 1 https://github.com/oopsvananh/deep_learning-waste_classification.git
%cd /content/deep_learning-waste_classification

In [ ]:
# Colab đã có PyTorch; chỉ cài các thư viện còn lại.
%pip install -q pandas scikit-learn pillow

import torch

assert torch.cuda.is_available(), "Hãy bật T4 GPU trong Runtime settings."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Dùng model, test loader và hàm đánh giá có sẵn trong repository.
from models.model_2.train_complex_cnn import ComplexCNN, classification_metrics, evaluate
from preprocess.preprocessing import CLASSES, compute_class_weights, make_dataloaders

_, _, test_loader = make_dataloaders()
model = ComplexCNN(num_classes=len(CLASSES)).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=compute_class_weights().to(device))

In [ ]:
# Nạp trọng số tốt nhất đã train, không cần train lại.
checkpoint = torch.load(
    "weights/complex_cnn_best_epoch187.pth",
    map_location=device,
    weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"])
print("Đã nạp checkpoint epoch", checkpoint["epoch"])

In [ ]:
# Đánh giá toàn bộ tập test và in bốn chỉ số chính.
test_result = evaluate(model, test_loader, loss_fn, device)
metrics = classification_metrics(test_result)

print("Số ảnh test:", len(test_loader.dataset))
print(f"Accuracy : {metrics['accuracy']:.2%}")
print(f"Precision: {metrics['precision']:.2%}")
print(f"Recall   : {metrics['recall']:.2%}")
print(f"F1-score : {metrics['f1_score']:.2%}")